# BGG Ontology: From Natural Language to SPARQL

Two-call Claude pipeline:
1. User question â†’ Claude generates SPARQL
2. Execute SPARQL against GraphDB
3. Claude interprets results â†’ natural-language answer
 
**Prerequisites:** GraphDB running locally with the BGG ontology loaded (see Setup section below).
Set the `ANTHROPIC_API_KEY` environment variable before starting Jupyter.


## Setup

Load data into GraphDB before running queries:
1. Create a repository named `bgg` (or change `REPO_NAME` below)
2. Import `bgg_kaggle2025.ttl` into the default graph (126k games, 1.3M triples)
3. Optionally import `fake_players.ttl` into the same default graph (200 fake players)


## Configuration

In [19]:
import os
import json
import textwrap
from pathlib import Path
import requests
import anthropic

# --- GraphDB ---
GRAPHDB_URL     = "http://localhost:7200"
REPO_NAME       = "bgg"                     # change to match your repository name
SPARQL_ENDPOINT = f"{GRAPHDB_URL}/repositories/{REPO_NAME}"

# --- Anthropic ---
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
if not ANTHROPIC_API_KEY:
    # Fallback: read from .env file (useful on Windows where env vars need a new process)
    _env_path = Path(os.getcwd()) / ".env"
    if _env_path.exists():
        for _line in _env_path.read_text().splitlines():
            if _line.startswith("ANTHROPIC_API_KEY="):
                ANTHROPIC_API_KEY = _line.split("=", 1)[1].strip().strip('"').strip("'")
                break

assert ANTHROPIC_API_KEY, (
    "ANTHROPIC_API_KEY not found. Either:\n"
    "  1. Restart VS Code after setting it in Windows environment settings, or\n"
    "  2. Create a .env file in this directory: ANTHROPIC_API_KEY=sk-ant-..."
)

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
MODEL  = "claude-opus-4-7"

print(f"GraphDB endpoint: {SPARQL_ENDPOINT}")
print(f"Model: {MODEL}")

GraphDB endpoint: http://localhost:7200/repositories/bgg
Model: claude-opus-4-7


## System Prompts

In [20]:
# BQ7
SYSTEM_SPARQL = """\
You are a SPARQL expert for a BoardGameGeek (BGG) knowledge graph stored in GraphDB.
Given a natural-language question, generate a single SPARQL 1.1 SELECT query that answers it.

## Board Game Knowledge Graph — Ontology Schema

### Namespaces (include in every query)
    PREFIX bgg:   <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX trenj: <https://vejdemo.se/boardgames/threnjen#>
    PREFIX svj:   <https://vejdemo.se/boardgames#>
    PREFIX fake:  <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf:   <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX skos:  <http://www.w3.org/2004/02/skos/core#>
    PREFIX xsd:   <http://www.w3.org/2001/XMLSchema#>

### Classes
- `bgg:Game` — A board game, tabletop game, or card game listed on BoardGameGeek — also called a title, game, or tabletop title.
- `bgg:Creator` — A person who designed, created, invented, authored, developed, or came up with a board game — also called a game designer, co-designer, author, or maker.
- `bgg:Category` — A thematic grouping, genre, type, style, or classification of a board game as defined by BGG — e.g. "Fantasy", "Card Game", "War Game". (see controlled vocabulary below)
- `bgg:Mechanic` — A gameplay mechanic, mechanism, rule system, or game system describing how a game is played — e.g. "Worker Placement", "Deck Building". Also called a gameplay element or game rule. (see controlled vocabulary below)
- `bgg:Publisher` — A company, studio, or individual that published, produced, released, or manufactured a board game.
- `bgg:Size` — The physical box size or weight class of a game: small (light, travel-sized), medium, or large (heavy, complex box).
- `bgg:MentalLoad` — The cognitive difficulty, complexity, or how hard a game is to learn and play: easy, moderate, or difficult.
- `bgg:Player` — A person who plays, owns, or collects board games — a gamer, board game player, or collector.
- `bgg:PlayerOpinion` — One player's personal opinion, review, rating, or assessment of a specific game.
- `trenj:Theme` — A thematic grouping, genre, topic, type, or what the game is about, as classified by the threnjen board game database — e.g. "Fantasy", "Steampunk", "Pirates". (see controlled vocabulary below)

### Datatype Properties on bgg:Game
- `bgg:hasName` xsd:string — The title, name, or what the game is called.
- `bgg:hasID` xsd:int — The BGG numeric ID, identifier, or BGG number.
- `bgg:hasDescription` xsd:string — A prose description, summary, overview, blurb, or "about" text for the game.
- `bgg:hasYearPublished` xsd:int — The year the game was published, released, came out, or first appeared.
- `bgg:hasRating` xsd:double — The average community rating or score on a 1–10 scale.
- `bgg:hasGeekRating` xsd:double — The BGG Geek Rating or BGG score.
- `bgg:ratingFromTime` xsd:int — The year the rating data was collected (2022 or 2025).
- `bgg:hasMinPlayers` xsd:int — The minimum number of players needed or required to play.
- `bgg:hasMaxPlayers` xsd:int — The maximum number of players supported, allowed, or accommodated.
- `bgg:hasBestNumPlayers` xsd:int — The optimal, recommended, or best player count.
- `bgg:hasMinGameTime` xsd:int — The minimum playtime, shortest duration, or quickest play time in minutes.
- `bgg:hasMaxGameTime` xsd:int — The maximum playtime, longest duration, or upper bound on play time in minutes.
- `bgg:hasMinRecAge` xsd:int — The minimum recommended age, age rating, or youngest suitable age in years.
- `bgg:isFullyEnriched` xsd:boolean — Whether the game has complete structured data: mechanics, categories, themes, creators, and publishers.

### Object Properties on bgg:Game
- `bgg:hasURL` URI — The BGG page link, web address, or URL for the game on BoardGameGeek.
- `bgg:hasThumbnail` URI — A thumbnail image, picture, cover art, or photo URL.
- `bgg:hasCategory` bgg:Category — The category, genre, type, or classification of the game. (match via rdfs:label)
- `bgg:hasMechanic` bgg:Mechanic — The gameplay mechanic, mechanism, or game system. (match via rdfs:label)
- `bgg:hasCreator` bgg:Creator — The designer, creator, or author of the game. (match via rdfs:label)
- `bgg:hasPublisher` bgg:Publisher — The publisher, company, or studio that produced the game. (match via rdfs:label)
- `bgg:hasSize` bgg:Size — box size: bgg:small / bgg:medium / bgg:large
- `trenj:hasTheme` trenj:Theme — The theme, topic, setting, or what the game is about. (match via rdfs:label)

### Object Properties on bgg:Player / bgg:PlayerOpinion
- `bgg:hasOwnershipOf` — A game that the player owns, has, or keeps in their collection.
- `bgg:hasOpinionHolder` — The player who holds, gave, or wrote this opinion or review.
- `bgg:hasOpinionOf` — The game this opinion, review, or rating is about.
- `bgg:hasMentalLoad` — The cognitive difficulty or complexity this player assigned: bgg:easy / bgg:moderate / bgg:difficult.
- `bgg:hasPlayerRatingOpinion` xsd:decimal — A player's personal rating, score, or how much they liked a game on a 1–10 scale.
- `bgg:likesCategory` — A category, genre, or type the player likes, enjoys, or prefers.
- `bgg:likesMechanic` — A mechanic or game system the player likes, enjoys, or prefers.

### Controlled Vocabulary — exact rdfs:label values

Use the vocabulary lists to identify the right concept name, then filter with CONTAINS(LCASE(str(?label)), LCASE("term")).
This handles variant and overlapping labels — e.g. CONTAINS("action point") matches both "Action Points" and "Action Point Allowance System".
Vocabulary instances also carry skos:altLabel for common synonyms and abbreviations (e.g. "WW1", "sci-fi", "co-op").
Use the rdfs:label|skos:altLabel property path to match both: ?v rdfs:label|skos:altLabel ?label .

**bgg:Category labels (89):**
"Abstract Strategy", "Action Dexterity", "Adventure", "Age of Reason", "American Civil War", "American Indian Wars", "American Revolutionary War", "American West", "Ancient", "Animals", "Arabian", "Aviation / Flight", "Aviation Flight", "Bluffing", "Book", "Card Game", "Childrens Game", "City Building", "Civil War", "Civilization", "Collectible Components", "Comic Book / Strip", "Comic Book Strip", "Deduction", "Dice", "Economic", "Educational", "Electronic", "Environmental", "Expansion for Base-game", "Exploration", "Fantasy", "Farming", "Fighting", "Game System", "Horror", "Humor", "Industry / Manufacturing", "Industry Manufacturing", "Korean War", "Mafia", "Math", "Mature / Adult", "Mature Adult", "Maze", "Medical", "Medieval", "Memory", "Miniatures", "Modern Warfare", "Movies / TV / Radio theme", "Movies TV Radio theme", "Murder Mystery", "Music", "Mythology", "Napoleonic", "Nautical", "Negotiation", "Novel-based", "Number", "Party Game", "Pike and Shot", "Pirates", "Political", "Post-Napoleonic", "Prehistoric", "Print & Play", "Puzzle", "Racing", "Real-Time", "Real-time", "Religious", "Renaissance", "Science Fiction", "Space Exploration", "Spies / Secret Agents", "Sports", "Territory Building", "Trains", "Transportation", "Travel", "Trivia", "Video Game Theme", "Vietnam War", "Wargame", "Word Game", "World War I", "World War II", "Zombies"

**bgg:Mechanic labels (176):**
"Acting", "Action Drafting", "Action Movement Programming", "Action Point Allowance System", "Action Points", "Action Queue", "Action Retrieval", "Action Timer", "Action/Event", "Advantage Token", "Alliances", "Area Control Area Influence", "Area Enclosure", "Area Majority / Influence", "Area Movement", "Area-Impulse", "Auction/Bidding", "Auction / Bidding", "Automatic Resource Growth", "Betting and Bluffing", "Betting / Wagering", "Bias", "Bingo", "Bribery", "Campaign / Battle Card Driven", "Campaign Battle Card Driven", "Card Play Conflict Resolution", "Catch the Leader", "Chaining", "Chit-Pull System", "Co-operative Play", "Command Cards", "Commodity Speculation", "Communication Limits", "Connections", "Contracts", "Cooperative Game", "Crayon Rail System", "Critical Hits and Failures", "Cube Tower", "Deck Construction", "Deck Pool Building", "Deck, Bag, and Pool Building", "Deduction", "Delayed Purchase", "Dexterity", "Dice Rolling", "Die Icon Resolution", "Different Dice Movement", "Drafting (altLabel: Card Drafting)", "Elapsed Real Time Ending", "Enclosure", "End Game Bonuses", "Events", "Finale Ending", "Follow", "Force Commitment", "Grid Coverage", "Grid Movement", "Hand Management", "Hex-and-Counter", "Hexagon Grid", "Hidden Movement", "Hidden Roles", "Hidden Victory Points", "Highest-Lowest Scoring", "I Cut, You Choose", "Impulse Movement", "Income", "Increase Value of Unchosen Resources", "Induction", "Interrupts", "Investment", "Kill Steal", "King of the Hill", "Ladder Climbing", "Layering", "Legacy Game", "Line Drawing", "Line of Sight", "Loans", "Lose a Turn", "Mancala", "Map Addition", "Map Deformation", "Map Reduction", "Market", "Matching", "Measurement Movement", "Melding and Splaying", "Memory", "Minimap Resolution", "Modular Board", "Move Through Deck", "Movement Points", "Movement Template", "Moving Multiple Units", "Multiple Maps", "Narrative Choice / Paragraph", "Negotiation", "Network and Route Building", "Once-Per-Game Abilities", "Order Counters", "Ownership", "Paper-and-Pencil", "Partnerships", "Passed Action Token", "Pattern Building", "Pattern Movement", "Pattern Recognition", "Physical", "Physical Removal", "Pick-up and Deliver", "Pieces as Map", "Player Elimination", "Player Judge", "Point to Point Movement", "Predictive Bid", "Press Your Luck", "Prisoner's Dilemma", "Programmed Movement", "Push Your Luck", "Race", "Random Production", "Ratio / Combat Results Table", "Re-rolling and Locking", "Real-Time", "Real-time", "Relative Movement", "Resource to Move", "Rock-Paper-Scissors", "Role Playing", "Roles with Asymmetric Information", "Roll / Spin and Move", "Roll Spin and Move", "Rondel", "RouteNetwork Building", "Scenario / Mission / Campaign Game", "Score-and-Reset Game", "Secret Unit Deployment", "Semi-Cooperative Game", "Set Collection", "Simulation", "Simultaneous Action Selection", "Singing", "Single Loser Game", "Slide/Push", "Solo / Solitaire Game", "Speed Matching", "Square Grid", "Stat Check Resolution", "Static Capture", "Stock Holding", "Storytelling", "Sudden Death Ending", "Tableau Building", "Take That", "Targeted Clues", "Team-Based Game", "Tech Trees / Tech Tracks", "Three Dimensional Movement", "Tile Placement", "Time Track", "Track Movement", "Trading", "Traitor Game", "Trick-taking", "Tug of War", "Variable Phase Order", "Variable Player Powers", "Variable Set-up", "Victory Points as a Resource", "Voting", "Worker Placement", "Zone of Control"

**trenj:Theme labels (222):**
"Adventure", "Age of Reason", "American Civil War", "American Indian Wars", "American Revolutionary War", "American West", "Ancient", "Animals", "Arabian", "Aviation / Flight", "Aviation Flight", "City Building", "Civil War", "Civilization", "Comic Book / Strip", "Comic Book Strip", "Economic", "Environmental", "Fantasy", "Farming", "Fighting", "Horror", "Humor", "Industry / Manufacturing", "Industry Manufacturing", "Korean War", "Mafia", "Math", "Mature / Adult", "Mature Adult", "Maze", "Medical", "Medieval", "Modern Warfare", "Movies / TV / Radio theme", "Movies TV Radio theme", "Murder Mystery", "Music", "Mythology", "Napoleonic", "Nautical", "Novel-based", "Number", "Pike and Shot", "Pirates", "Political", "Post-Napoleonic", "Prehistoric", "Racing", "Religious", "Renaissance", "Science Fiction", "Space Exploration", "Spies / Secret Agents", "Sports", "African Americans", "Airships / Blimps / Dirigibles / Zeppelins", "Alchemy", "Alternate History", "Amusement Parks / Theme Parks", "Animal Battles", "Anime / Manga", "Anthropomorphic Animals", "Apache Tribes", "Arcade Video Games", "Archaeology / Paleontology", "Art", "Astrology", "Astronomy", "Attorneys / Courts", "Automotive Industry", "Aztecs", "Bacteria", "Battle Royal", "Beaches", "Biology", "Boardgaming", "Books / Libraries", "Camping", "Canals", "Cannibals / Cannibalis", "Care Bears", "Cemeteries / Graveyards", "Cereal Games", "Chemistry", "Chernobyl", "Chibis", "Chivalry / Jousting / Tournaments (Medieval Europe)", "Circus", "City", "Climate Chang", "Colonial", "Computer / Information Technology Industry", "Construction", "Cruise ships", "Cthulhu Mythos", "Cyberpunk", "Deserts", "Dieselpunk", "Disney Theme Parks", "Dolls", "Dreams / Nightmares", "Druids", "Earthquakes", "Ecology", "Evolution", "FIFA World Cup", "Fantasy Sports", "Fashion", "Fictional Games", "Firefighting", "Flags identification", "Floating islands in the sky", "Flowers", "Food / Cooking", "French Foreign Legion", "Fruit", "Gardening", "Geocaching", "Gladiators", "Hackers", "Hanseatic Leagu", "Helicopters", "Hell", "Hike-Hiking", "Hot Air Balloons", "Inuit Peoples", "Jail / Prison (Modern)", "Jewelry", "Journalis", "Kaiju", "King Arthur / The Knights of the Round Table / Camelot", "Knights Templar", "Latin American Political Games", "Love / Romanc", "Mad Science / Mad Scientist", "Mail / Stamps / The Post Offic", "Mayans", "Mech Warfar", "Memes", "Mining", "Motorcycles", "Movie Industry", "Movies", "Mushrooms", "Music Making & Makers", "My Best Lif", "Mystery / Cri", "Māori", "Native Americans / First Peoples", "Natur", "Ninjas", "Nuclear option", "Oil / Gas / Petroleu", "Painting / Paintings", "Perfu", "Photography", "Polic", "Post-Apocalyptic", "Psychic Powers", "Psychology", "Pub, Bars, Bistros...", "Pulp", "Retro", "Rivers", "Robots", "Romance of the Three Kingdoms", "Rubik's Cub", "Safaris", "Samurai", "School / College / University", "Sci-Fi Sports", "Scienc", "Sewing / Knitting / Cloth-Making", "Sieg", "Silk Road", "Spanish Political Games", "Spooky Old Houses", "Steampunk", "Submarines", "Superheroes", "Survival", "TV Detectives", "Teaching Programming", "Television (TV) Industry", "Templ", "Tiki Cultur", "Time Travel", "Traffic / Driving", "Trash / Garbag", "Trees and Forests", "Tropical", "Tropical Islands", "Trucks", "UFOs", "US National Parks", "Under the Sea", "Video Game Theme: Carmen Sandiego", "Video Game Theme: Doo", "Video Game Theme: Final Fantasy", "Video Game Theme: Minecraft", "Video Game Theme: Nintendo", "Video Game Theme: Pokémon", "Video Game Theme: Resident Evil", "Video Game Theme: SEGA", "Video Game Theme: Super Mario Bros.", "Video Game Theme: Tetris", "Video Game Theme: The Oregon Trail", "Vikings", "Villainy", "Volcanoes", "Weather", "Witches", "Trains", "Transportation", "Travel", "Trivia", "Video Game Theme", "Vietnam War", "World War I", "World War II", "Zombies"

### IMPORTANT DATA NOTES
1. Only 21,379 games have `bgg:isFullyEnriched true` and carry mechanics/categories/themes/creators/publishers.
   The other ~105k games have only: name, description, year, ratings, thumbnail, URL.
2. Always use an exact label from the controlled vocabulary lists above when filtering by category, mechanic, or theme.
3. `bgg:hasName` values are xsd:string. ALWAYS match game names with FILTER(CONTAINS(LCASE(str(?name)), LCASE("term"))). Never use exact literal match — games often have subtitles or variant spellings.
4. For case-insensitive name search: FILTER(CONTAINS(LCASE(str(?name)), "catan")).
5. Fake player data (200 players, namespace fake:) and Susanne's data (namespace svj:) are in the same graph.
6. Susanne Vejdemo's IRI is svj:SusanneVejdemo — use it directly (e.g. svj:SusanneVejdemo bgg:hasOwnershipOf ?game). Do NOT guess other svj: IRIs; look up other players by rdfs:label.
7. For numeric FILTER expressions, use bare integers/decimals — not typed string literals:
   FILTER(?year = 2000)  FILTER(?rating > 8.0)  FILTER(?min <= 4 && ?max >= 4)
   Never write FILTER(?year = "2000"^^xsd:int) — the bare form matches all numeric XSD types.

8. SPARQL 1.1 property paths enable transitive traversal without recursion:
   - `(p/q)+` follows a path of p then q, one-or-more times
   - `p*` means zero-or-more hops along p
   - `^p` is the inverse of p (follows p backwards)
   - Example: `svj:SusanneVejdemo (bgg:hasOwnershipOf/^bgg:hasOwnershipOf)+ ?neighbor`
     finds ALL players reachable through any chain of shared game ownership, any depth
   - Example: `?game bgg:reimplements* bgg:9209` finds all games in the Ticket to Ride
     family (the original plus reimplementations of reimplementations, etc.)
   - `bgg:reimplements` — subject=newer reimplementation, object=original game

### Example SPARQL Queries

Find a game by name (CONTAINS — preferred for all user-provided names):
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    SELECT ?name ?year ?rating WHERE {
      ?game a bgg:Game ;
            bgg:hasName ?name ;
            bgg:hasYearPublished ?year ;
            bgg:hasRating ?rating .
      FILTER(CONTAINS(LCASE(str(?name)), LCASE("catan")))
    }
    ORDER BY DESC(?rating) LIMIT 10

Top-rated games for 2 players:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    SELECT ?name ?rating WHERE {
      ?game a bgg:Game ;
            bgg:hasName ?name ;
            bgg:hasRating ?rating ;
            bgg:hasMinPlayers ?min ;
            bgg:hasMaxPlayers ?max .
      FILTER(?min <= 2 && ?max >= 2)
    }
    ORDER BY DESC(?rating) LIMIT 10

Games with Worker Placement mechanic (CONTAINS for label matching):
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?name ?rating WHERE {
      ?game a bgg:Game ;
            bgg:hasName ?name ;
            bgg:hasMechanic ?mech ;
            bgg:hasRating ?rating .
      ?mech rdfs:label ?mechLabel .
      FILTER(CONTAINS(LCASE(str(?mechLabel)), LCASE("worker placement")))
    }
    ORDER BY DESC(?rating) LIMIT 10

Fake player opinions:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?playerName ?gameName ?personalRating WHERE {
      ?op a bgg:PlayerOpinion ;
          bgg:hasOpinionHolder ?player ;
          bgg:hasOpinionOf ?game ;
          bgg:hasPlayerRatingOpinion ?personalRating .
      ?player rdfs:label ?playerName .
      ?game bgg:hasName ?gameName .
      FILTER(?personalRating = 10)
    } LIMIT 20

Games about a topic — always check both Category and Theme using UNION with CONTAINS:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX trenj: <https://vejdemo.se/boardgames/threnjen#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT DISTINCT ?name ?rating WHERE {
      ?game a bgg:Game ;
            bgg:hasName ?name ;
            bgg:hasRating ?rating .
      { ?game bgg:hasCategory ?tag . ?tag rdfs:label ?tagLabel .
        FILTER(CONTAINS(LCASE(str(?tagLabel)), LCASE("fantasy"))) }
      UNION
      { ?game trenj:hasTheme ?tag . ?tag rdfs:label ?tagLabel .
        FILTER(CONTAINS(LCASE(str(?tagLabel)), LCASE("fantasy"))) }
    }
    ORDER BY DESC(?rating) LIMIT 10

Games matching a concept across both categories and mechanics — UNION with altLabel property path:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    SELECT DISTINCT ?name ?rating WHERE {
      ?game a bgg:Game ;
            bgg:hasName ?name ;
            bgg:hasRating ?rating .
      { ?game bgg:hasCategory ?v } UNION { ?game bgg:hasMechanic ?v }
      ?v rdfs:label|skos:altLabel ?label .
      FILTER(CONTAINS(LCASE(str(?label)), LCASE("deduction")))
    }
    ORDER BY DESC(?rating) LIMIT 10

Game name and creator if known — OPTIONAL for enriched-only properties:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?name ?year ?rating ?creatorName WHERE {
      ?game a bgg:Game ;
            bgg:hasName ?name ;
            bgg:hasYearPublished ?year ;
            bgg:hasRating ?rating .
      OPTIONAL {
        ?game bgg:hasCreator ?creator .
        ?creator rdfs:label ?creatorName .
      }
    }
    ORDER BY DESC(?rating) LIMIT 10

Most common mechanics — GROUP BY with HAVING:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?mechName (COUNT(?game) AS ?gameCount) (AVG(?rating) AS ?avgRating) WHERE {
      ?game a bgg:Game ;
            bgg:hasMechanic ?mech ;
            bgg:hasRating ?rating .
      ?mech rdfs:label ?mechName .
    }
    GROUP BY ?mechName
    HAVING (COUNT(?game) > 10)
    ORDER BY DESC(?gameCount) LIMIT 10

Player recommendations — games matching liked categories not yet owned:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT DISTINCT ?name ?rating WHERE {
      ?player a bgg:Player ;
              rdfs:label "Alice"@en ;
              bgg:likesCategory ?cat .
      ?game a bgg:Game ;
            bgg:hasName ?name ;
            bgg:hasRating ?rating ;
            bgg:hasCategory ?cat .
      FILTER NOT EXISTS { ?player bgg:hasOwnershipOf ?game }
    }
    ORDER BY DESC(?rating) LIMIT 10

Find games similar to X — shared mechanics, themes, and categories, ranked by overlap count:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX trenj: <https://vejdemo.se/boardgames/threnjen#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT DISTINCT ?name ?rating (COUNT(DISTINCT ?shared) AS ?overlap) WHERE {
      ?ref a bgg:Game ; bgg:hasName ?refName .
      FILTER(CONTAINS(LCASE(str(?refName)), LCASE("dominion")))
      { ?ref bgg:hasMechanic ?shared . ?game bgg:hasMechanic ?shared . }
      UNION
      { ?ref trenj:hasTheme ?shared . ?game trenj:hasTheme ?shared . }
      UNION
      { ?ref bgg:hasCategory ?shared . ?game bgg:hasCategory ?shared . }
      ?game a bgg:Game ;
            bgg:hasName ?name ;
            bgg:hasRating ?rating .
      FILTER(?game != ?ref)
      FILTER(!CONTAINS(LCASE(str(?name)), LCASE("dominion")))
      # Add extra constraints here, e.g. require an additional mechanic:
      # ?game bgg:hasMechanic ?extraMech . ?extraMech rdfs:label|skos:altLabel ?eLabel .
      # FILTER(CONTAINS(LCASE(str(?eLabel)), LCASE("cooperative")))
    }
    GROUP BY ?name ?rating
    ORDER BY DESC(?overlap) DESC(?rating) LIMIT 15

Transitive community detection — ALL players reachable through shared ownership chains:
    PREFIX bgg:  <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX svj:  <https://vejdemo.se/boardgames#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT DISTINCT ?memberLabel WHERE {
      svj:SusanneVejdemo (bgg:hasOwnershipOf/^bgg:hasOwnershipOf)+ ?member .
      ?member rdfs:label ?memberLabel .
      FILTER(?member != svj:SusanneVejdemo)
    }
    ORDER BY ?memberLabel LIMIT 200

Game family traversal — all games reimplementing an original, at any depth:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    SELECT ?gameName ?gameRating WHERE {
      ?game bgg:reimplements* bgg:9209 .
      ?game bgg:hasName ?gameName .
      OPTIONAL { ?game bgg:hasRating ?gameRating }
    }
    ORDER BY DESC(?gameRating)

Transitive community detection -- ALL players reachable through shared ownership chains:
    PREFIX bgg:  <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX svj:  <https://vejdemo.se/boardgames#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT DISTINCT ?memberLabel WHERE {
      svj:SusanneVejdemo (bgg:hasOwnershipOf/^bgg:hasOwnershipOf)+ ?member .
      ?member rdfs:label ?memberLabel .
      FILTER(?member != svj:SusanneVejdemo)
    }
    ORDER BY ?memberLabel LIMIT 200

Game family traversal -- all games reimplementing an original, at any depth:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    SELECT ?gameName ?gameRating WHERE {
      ?game bgg:reimplements* bgg:9209 .
      ?game bgg:hasName ?gameName .
      OPTIONAL { ?game bgg:hasRating ?gameRating }
    }
    ORDER BY DESC(?gameRating)


Rules:
- Always include all required PREFIX declarations.
- Return ONLY the raw SPARQL query with no explanation, no markdown fences, no commentary.
- Use OPTIONAL { } for properties that may not exist on all games (mechanics, categories, themes, creators, publishers only exist for the 21,379 fully enriched games).
- Use ORDER BY DESC(?rating) and LIMIT when asked for 'best', 'top', or 'most popular'.
- When filtering by category, mechanic, or theme, use CONTAINS(LCASE(str(?label)), LCASE("term")) rather than exact match, to handle variant and overlapping labels. Use the vocabulary lists to choose an appropriate search term.
- When the user asks about topic, theme, genre, setting, or what a game is about, always query BOTH bgg:hasCategory and trenj:hasTheme using a UNION.
- When the user's term could describe either a category or a mechanic (e.g. "deduction game", "bluffing game", "cooperative game", "racing game"), also UNION across bgg:hasCategory and bgg:hasMechanic using the rdfs:label|skos:altLabel property path.
- When altLabels are present, use rdfs:label|skos:altLabel ?label in place of rdfs:label ?label to match both standard and alternative labels.
- For player count: "N players" / "for N players" / "playable by N" means N is within the valid range: FILTER(?min <= N && ?max >= N). "Exactly N players" / "strictly N" / "only N players" means the game is designed for that count only: FILTER(?min = N && ?max = N). "At least N players" means FILTER(?min <= N). "At most N players" / "up to N players" means FILTER(?max <= N).
- For any user-provided game name: always use FILTER(CONTAINS(LCASE(str(?name)), LCASE("term"))) — never exact literal match. When the question asks about a specific game (e.g. "max players for Catan"), use CONTAINS and then take MAX() or LIMIT 1 over all matching games.
- Do not use DESCRIBE or CONSTRUCT — only SELECT queries.
- For "similar to X", "I like X", or "I enjoy X" recommendation queries: look up X's mechanics, themes, and categories; find candidates sharing those properties via three-way UNION (hasMechanic UNION hasTheme UNION hasCategory); rank by COUNT(DISTINCT ?shared) DESC then DESC(?rating); exclude the reference game with FILTER(?game != ?ref); apply additional constraints inside WHERE on top of the similarity join. If zero or very few results are found, report what is found and note the limitation.
- Avoid Cartesian products; use FILTER close to the relevant triple pattern.
"""

SYSTEM_RESULT = """\
You are a helpful assistant explaining SPARQL query results about board games.
Given a question and the raw SPARQL results (as JSON), write a clear, concise natural-language answer.
Focus on answering the question directly. If the results are empty, say so and suggest why.
If results_truncated is true, note that only the top 50 of result_count results are shown.
Format lists nicely. Round numbers to 2 decimal places. Keep your answer under 200 words.
"""

print(f"SPARQL system prompt: {len(SYSTEM_SPARQL)} chars")
print(f"Result system prompt: {len(SYSTEM_RESULT)} chars")

# Alternative label-matching method — exact match (swap in to compare against CONTAINS method above).
# To test: replace the vocabulary intro note, the two label-filtering examples, and the rule below.
PROMPT_ADDITION_ALTERNATIVE = """
# Vocabulary intro (replaces current CONTAINS note in SYSTEM_SPARQL):
When filtering by category, mechanic, or theme, use an exact label from these lists.
Map user terms to the closest matching label (e.g. "sci-fi" → "Science Fiction", "pirates" → "Pirates", "steampunk" → "Steampunk").

# Example — Worker Placement (exact match, replaces CONTAINS version):
      ?mech rdfs:label "Worker Placement"@en .

# Example — UNION topic query (exact match, replaces CONTAINS version):
      { ?game bgg:hasCategory ?tag . ?tag rdfs:label "Fantasy"@en }
      UNION
      { ?game trenj:hasTheme ?tag . ?tag rdfs:label "Fantasy"@en }

# Rule (replaces CONTAINS rule in Rules section):
- When filtering by category, mechanic, or theme, always use an exact label from the
  controlled vocabulary lists. Map user terms to the closest matching label.
"""

SPARQL system prompt: 22378 chars
Result system prompt: 455 chars


## Pipeline Functions

In [21]:
import json as _json
from datetime import datetime as _dt

def _log_qa(question_id: str, question: str, answer: str, agent: str) -> None:
    entry = {
        "timestamp": _dt.now().isoformat(timespec="seconds"),
        "agent": agent,
        "question_id": question_id,
        "question": question,
        "answer": answer,
        "score": None,
        "comment": "",
    }
    with open("qa_log.jsonl", "a", encoding="utf-8") as _f:
        _f.write(_json.dumps(entry, ensure_ascii=False) + "\n")

MAX_RETRIES = 2


def generate_sparql(question: str, error_context: str = None) -> str:
    """Call 1: natural language -> SPARQL query. Pass error_context to repair a bad query."""
    user_content = question
    if error_context:
        nl = chr(10)
        user_content = (nl * 2).join([
            question,
            f"Your previous query failed with this GraphDB error: {error_context}",
            "Please fix the SPARQL query.",
        ])
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=[
            {
                "type": "text",
                "text": SYSTEM_SPARQL,
                "cache_control": {"type": "ephemeral"},
            }
        ],
        messages=[{"role": "user", "content": user_content}],
    )
    query = response.content[0].text.strip()
    # Strip markdown fences if present
    if query.startswith("```"):
        query = chr(10).join(
            l for l in query.splitlines() if not l.startswith("```")
        ).strip()
    return query


def execute_sparql(query: str) -> dict:
    """Execute a SPARQL SELECT against GraphDB; return parsed JSON results."""
    resp = requests.post(
        SPARQL_ENDPOINT,
        data={"query": query},
        headers={"Accept": "application/sparql-results+json"},
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()


def format_results_table(results: dict, max_rows: int = 20) -> str:
    """Format SPARQL JSON results as a markdown table (values truncated at 80 chars)."""
    bindings = results.get("results", {}).get("bindings", [])
    vars_    = results.get("head",    {}).get("vars",     [])
    if not bindings or not vars_:
        return "(no results)"

    def cell(b, v):
        val = b.get(v, {}).get("value", "")
        return val[:80] + "..." if len(val) > 80 else val

    rows  = [[cell(b, v) for v in vars_] for b in bindings]
    shown = rows[:max_rows]
    col_w = [max(len(v), max(len(r[i]) for r in shown)) for i, v in enumerate(vars_)]

    def md_row(cells):
        return "| " + " | ".join(c.ljust(col_w[i]) for i, c in enumerate(cells)) + " |"

    header  = md_row(vars_)
    divider = "| " + " | ".join("-" * col_w[i] for i in range(len(vars_))) + " |"
    lines   = [header, divider] + [md_row(r) for r in shown]
    if len(bindings) > max_rows:
        lines.append(f"*(showing {max_rows} of {len(bindings)} rows)*")
    return chr(10).join(lines)


def interpret_results(question: str, query: str, results: dict) -> str:
    """Call 2: SPARQL results -> natural-language answer."""
    bindings = results.get("results", {}).get("bindings", [])
    vars_    = results.get("head",    {}).get("vars",     [])
    truncated = len(bindings) > 50

    payload = {
        "question": question,
        "sparql_query": query,
        "result_variables": vars_,
        "result_count": len(bindings),
        "results_truncated": truncated,
        "results": bindings[:50],
    }

    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=SYSTEM_RESULT,
        messages=[
            {
                "role": "user",
                "content": json.dumps(payload, ensure_ascii=False),
            }
        ],
    )
    return response.content[0].text.strip()


def ask(question: str, show_query: bool = True, question_id: str = None) -> str:
    """Full pipeline: question -> SPARQL -> execute (with retry) -> natural-language answer."""
    print(f"Question: {question}")
    print()

    query = generate_sparql(question)
    for attempt in range(MAX_RETRIES + 1):
        if show_query:
            label = f"Generated SPARQL (attempt {attempt + 1}):" if attempt > 0 else "Generated SPARQL:"
            print(label)
            print("-" * 60)
            print(query)
            print("-" * 60)
            print()

        try:
            results = execute_sparql(query)
            break
        except requests.HTTPError as e:
            error_msg = e.response.text[:500]
            if attempt < MAX_RETRIES:
                print(f"SPARQL error (attempt {attempt + 1}), retrying...")
                print(error_msg[:120])
                print()
                query = generate_sparql(question, error_msg)
            else:
                print(f"SPARQL execution failed after {MAX_RETRIES + 1} attempts.")
                print(error_msg)
                return ""

    n = len(results.get("results", {}).get("bindings", []))
    print(f"Results: {n} row(s)" + (" (truncated to 50 for interpretation)" if n > 50 else ""))
    print()
    print(format_results_table(results))
    print()

    answer = interpret_results(question, query, results)
    print("Answer:")
    print(textwrap.fill(answer, width=80))
    print()
    if question_id:
        _log_qa(question_id, question, answer, "sparql")
    return answer

## Test GraphDB Connectivity

In [22]:
test_query = "SELECT (COUNT(*) AS ?n) WHERE { ?s ?p ?o }"
try:
    result = execute_sparql(test_query)
    count = result["results"]["bindings"][0]["n"]["value"]
    print(f"GraphDB is reachable. Total triples: {count}")
except Exception as e:
    print(f"Cannot reach GraphDB: {e}")
    print(f"Make sure GraphDB is running at {GRAPHDB_URL}")
    print(f"and a repository named '{REPO_NAME}' exists with the TTL loaded.")


GraphDB is reachable. Total triples: 579724


## Example Queries

Run any cell below, or write your own at the bottom.


In [23]:
ask("What are the top 10 highest-rated board games of all time?")

Question: What are the top 10 highest-rated board games of all time?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
SELECT ?name ?year ?rating WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasYearPublished ?year ;
        bgg:hasRating ?rating .
}
ORDER BY DESC(?rating) LIMIT 10
------------------------------------------------------------

Results: 10 row(s)

| name                                                  | year | rating |
| ----------------------------------------------------- | ---- | ------ |
| TerroriXico                                           | 2018 | 9.43   |
| Meeple Quest                                          | 2021 | 9.32   |
| The Fantasy Trip: Legacy Edition                      | 2019 | 9.29   |
| Wings of the Motherland                               | 2019 | 9.17   |
| 1985: Deadly Northern Lights                          | 2020 | 9.17 

"# Top 10 Highest-Rated Board Games\n\nHere are the highest-rated board games in the dataset:\n\n1. **TerroriXico** (2018) — 9.43\n2. **Meeple Quest** (2021) — 9.32\n3. **The Fantasy Trip: Legacy Edition** (2019) — 9.29\n4. **Wings of the Motherland** (2019) — 9.17\n5. **1985: Deadly Northern Lights** (2020) — 9.17\n6. **System Gateway** (fan expansion for Android: Netrunner) (2021) — 9.16\n7. **Global War 1939** (2011) — 9.09\n8. **Animator vs Animation** (2018) — 9.02\n9. **World in Flames Collector's Edition Deluxe** (2017) — 9.01\n10. **Urgency** (2020) — 9.00\n\nNote that these are relatively niche or recent titles rather than mainstream classics—likely because the query ranks purely by rating score without factoring in the number of ratings. Popular, well-established games tend to have slightly lower averages due to their broader audience."

In [24]:
ask("Which games use the Worker Placement mechanic and have a rating above 8?")

Question: Which games use the Worker Placement mechanic and have a rating above 8?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT DISTINCT ?name ?rating WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasMechanic ?mech ;
        bgg:hasRating ?rating .
  ?mech rdfs:label|skos:altLabel ?mechLabel .
  FILTER(CONTAINS(LCASE(str(?mechLabel)), LCASE("worker placement")))
  FILTER(?rating > 8.0)
}
ORDER BY DESC(?rating)
------------------------------------------------------------

Results: 35 row(s)

| name                                                                       | rating |
| -------------------------------------------------------------------------- | ------ |
| Anachrony: Infinity Box                                                    | 8.97   |
| 

"There are **35 Worker Placement games** with a rating above 8.0. Here are the top-rated picks:\n\n**Top 10 Highest-Rated:**\n1. Anachrony: Infinity Box — 8.97\n2. Trickerion: Collector's Edition — 8.76\n3. Everdell: Collector's Edition — 8.52\n4. The Castles of Burgundy — 8.44\n5. Dune: Imperium — 8.43\n6. Kanban EV — 8.41\n7. Lorenzo il Magnifico: Big Box — 8.40\n8. Ultimate Railroads — 8.39\n9. A Song of Ice & Fire: Night's Watch Starter Set — 8.37\n10. Viticulture: Complete Collector's Edition — 8.32\n\n**Other notable entries** include On Mars (8.18), A Feast for Odin (8.17), Barrage (8.16), Dwellings of Eldervale (8.13), Lost Ruins of Arnak (8.07), Orléans (8.05), Underwater Cities (8.05), Fields of Arle (8.03), The Gallerist (8.02), and Everdell (8.01).\n\nThe list is dominated by heavier euro-style strategy games, with several deluxe/big box editions ranking especially high. All 35 results are shown (no truncation)."

In [25]:
ask("What games can be played by exactly 2 players and take less than 30 minutes?")

Question: What games can be played by exactly 2 players and take less than 30 minutes?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
SELECT ?name ?rating ?maxTime WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasRating ?rating ;
        bgg:hasMinPlayers ?min ;
        bgg:hasMaxPlayers ?max ;
        bgg:hasMaxGameTime ?maxTime .
  FILTER(?min = 2 && ?max = 2)
  FILTER(?maxTime < 30)
}
ORDER BY DESC(?rating) LIMIT 20
------------------------------------------------------------

Results: 20 row(s)

| name                                                                                | rating | maxTime |
| ----------------------------------------------------------------------------------- | ------ | ------- |
| Revanche                                                                            | 8.46   | 20      |
| BattleCON: Wanderers of Indines                 

'Here are the top 20 highest-rated games designed strictly for 2 players with a maximum playtime under 30 minutes:\n\n1. **Revanche** — 8.46 (20 min)\n2. **BattleCON: Wanderers of Indines** — 8.36 (25 min)\n3. **Exceed: Street Fighter – Chun-Li Box** — 8.24 (25 min)\n4. **Exceed: Shovel Knight – Shadow Box** — 8.23 (25 min)\n5. **Exceed: Street Fighter – M. Bison Box** — 8.22 (25 min)\n6. **Unmatched: Little Red Riding Hood vs. Beowulf** — 8.11 (20 min)\n7. **Exceed: Shovel Knight – Hope Box** — 8.10 (25 min)\n8. **Unmatched: Robin Hood vs. Bigfoot** — 7.92 (20 min)\n9. **Onager** — 7.88 (20 min)\n10. **Exceed: Red Horizon – Satoshi, Mei Lien, Morathi, and Baelkhor** — 7.83 (20 min)\n11. **Exceed: Street Fighter – Ryu Box** — 7.83 (25 min)\n12. **Star Realms: Colony Wars** — 7.79 (20 min)\n13. **Cuttle** — 7.77 (5 min)\n14. **Exceed: Red Horizon (Kaden set)** — 7.72 (20 min)\n15. **Exceed: Red Horizon'

In [26]:
ask("What games did Uwe Rosenberg design?")

Question: What games did Uwe Rosenberg design?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT DISTINCT ?name ?year ?rating WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasCreator ?creator .
  ?creator rdfs:label ?creatorName .
  OPTIONAL { ?game bgg:hasYearPublished ?year . }
  OPTIONAL { ?game bgg:hasRating ?rating . }
  FILTER(CONTAINS(LCASE(str(?creatorName)), LCASE("Uwe Rosenberg")))
}
ORDER BY DESC(?rating)
------------------------------------------------------------

Results: 67 row(s) (truncated to 50 for interpretation)

| name                                                | year | rating |
| --------------------------------------------------- | ---- | ------ |
| A Feast for Odin                                    | 2016 | 8.17   |
| Fields of Arle                                      | 2014 | 8

'Uwe Rosenberg has designed 67 games in the database. Here are his top-rated titles (showing the top 50 of 67):\n\n**Top Heavyweight Euros:**\n- A Feast for Odin (2016) — 8.17\n- Fields of Arle (2014) — 8.03\n- Agricola (Revised Edition) (2016) — 7.95\n- Caverna: The Cave Farmers (2013) — 7.94\n- Agricola (2007) — 7.87\n- Hallertau (2020) — 7.86\n- Le Havre (2008) — 7.84\n- Ora et Labora (2011) — 7.68\n- Nusfjord (2017) — 7.64\n\n**Lighter / Family Favorites:**\n- Patchwork (2014) — 7.59\n- Glass Road (2013) — 7.42\n- At the Gates of Loyang (2009) — 7.36\n- Nova Luna (2019) — 7.27\n- Sagani (2020) — 7.10\n- Bohnanza (1997) — 7.07\n\n**Notable others:** Indian Summer, Cottage Garden, Reykholt, Spring Meadow, New York Zoo, Patchwork Doodle, and many Bohnanza spin-offs (Würfel Bohnanza, Bohnanza: The Duel, etc.).\n\nRosenberg is best known for heavy farming/economic Euros (the Agricola/Caverna/Le Havre family) plus the lighter Patchwork and Bohnanza series.'

In [27]:
ask("What games did Uwe Rossenberg design?")
### Intentional misspelling of name 

Question: What games did Uwe Rossenberg design?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT DISTINCT ?name ?year ?rating WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasCreator ?creator .
  ?creator rdfs:label ?creatorName .
  OPTIONAL { ?game bgg:hasYearPublished ?year }
  OPTIONAL { ?game bgg:hasRating ?rating }
  FILTER(CONTAINS(LCASE(str(?creatorName)), LCASE("rosenberg")))
}
ORDER BY DESC(?rating)
LIMIT 50
------------------------------------------------------------

Results: 50 row(s)

| name                                                | year | rating |
| --------------------------------------------------- | ---- | ------ |
| A Feast for Odin                                    | 2016 | 8.17   |
| Fields of Arle                                      | 2014 | 8.03   |
| Agricola (Revised Edition

'# Games Designed by Uwe Rosenberg\n\nUwe Rosenberg is one of the most prolific Eurogame designers, known especially for heavy farming/worker-placement games and the *Bohnanza* card game family. Here are his designs from the database, sorted by rating (showing the top 50 results):\n\n## Top-Rated Heavy Eurogames\n- **A Feast for Odin** (2016) – 8.17\n- **Fields of Arle** (2014) – 8.03\n- **Agricola (Revised Edition)** (2016) – 7.95\n- **Caverna: The Cave Farmers** (2013) – 7.94\n- **Agricola** (2007) – 7.87\n- **Hallertau** (2020) – 7.86\n- **Le Havre** (2008) – 7.84\n- **Ora et Labora** (2011) – 7.68\n- **Nusfjord** (2017) – 7.64\n\n## Lighter / Family Games\n- **Patchwork** (2014) – 7.59\n- **Glass Road** (2013) – 7.42\n- **Nova Luna** (2019) – 7.27\n- **Indian Summer** (2017) – 6.76\n- **Cottage Garden** (2016) – 6.85\n- **New York Zoo** (2020) – 7.04\n\n## The Bohnanza Family\n- **Bohnanza** (1997) – 7.07, plus many spin-offs (*Bohnanza: The Duel*, *Würfel Bohnanza*, *Ladybohn*, *M

In [28]:
ask("Which categories have the most games?")

Question: Which categories have the most games?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?catName (COUNT(?game) AS ?gameCount) WHERE {
  ?game a bgg:Game ;
        bgg:hasCategory ?cat .
  ?cat rdfs:label ?catName .
}
GROUP BY ?catName
ORDER BY DESC(?gameCount) LIMIT 20
------------------------------------------------------------

Results: 20 row(s)

| catName                | gameCount |
| ---------------------- | --------- |
| Card Game              | 6300      |
| Miniatures             | 1048      |
| Wargame                | 945       |
| Exploration            | 859       |
| Fantasy                | 717       |
| Puzzle                 | 651       |
| Print & Play           | 575       |
| Economic               | 554       |
| Fighting               | 518       |
| Territory Building     | 496       |
| Educationa

'# Top Board Game Categories\n\n**Card Game** dominates by a wide margin with **6,300 games**, far outpacing all other categories.\n\nHere are the top 20 categories by game count:\n\n| Rank | Category | Games |\n|------|----------|-------|\n| 1 | Card Game | 6,300 |\n| 2 | Miniatures | 1,048 |\n| 3 | Wargame | 945 |\n| 4 | Exploration | 859 |\n| 5 | Fantasy | 717 |\n| 6 | Puzzle | 651 |\n| 7 | Print & Play | 575 |\n| 8 | Economic | 554 |\n| 9 | Fighting | 518 |\n| 10 | Territory Building | 496 |\n| 11 | Educational | 476 |\n| 12 | Word Game | 473 |\n| 13 | Science Fiction | 464 |\n| 14 | Dice | 353 |\n| 15 | World War II | 340 |\n| 16 | Bluffing | 337 |\n| 17 | Collectible Components | 336 |\n| 18 | Medieval | 328 |\n| 19 | Party Game | 297 |\n| 20 | Adventure | 295 |\n\nCard Games alone represent roughly 6× more titles than the next category, reflecting how prolific card-based designs are in the hobby.'

In [29]:
ask("What games are in the Fantasy category with a geek rating above 7.5?")

Question: What games are in the Fantasy category with a geek rating above 7.5?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX trenj: <https://vejdemo.se/boardgames/threnjen#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT DISTINCT ?name ?geekRating WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasGeekRating ?geekRating .
  { ?game bgg:hasCategory ?tag . ?tag rdfs:label|skos:altLabel ?tagLabel .
    FILTER(CONTAINS(LCASE(str(?tagLabel)), LCASE("fantasy"))) }
  UNION
  { ?game trenj:hasTheme ?tag . ?tag rdfs:label|skos:altLabel ?tagLabel .
    FILTER(CONTAINS(LCASE(str(?tagLabel)), LCASE("fantasy"))) }
  FILTER(?geekRating > 7.5)
}
ORDER BY DESC(?geekRating) LIMIT 50
------------------------------------------------------------

Results: 26 row(s)

| name                                     | gee

"Found 26 Fantasy-themed games with a geek rating above 7.5:\n\n1. **Gloomhaven** – 8.34\n2. **War of the Ring: Second Edition** – 8.19\n3. **Spirit Island** – 8.14\n4. **Gloomhaven: Jaws of the Lion** – 8.14\n5. **Mage Knight: Ultimate Edition** – 7.96\n6. **Clank! Legacy: Acquisitions Incorporated** – 7.95\n7. **Arkham Horror: The Card Game** – 7.90\n8. **Root** – 7.89\n9. **Terra Mystica** – 7.88\n10. **Too Many Bones** – 7.87\n11. **Mage Knight Board Game** – 7.85\n12. **Everdell** – 7.83\n13. **Caverna: The Cave Farmers** – 7.75\n14. **Blood Rage** – 7.74\n15. **Cthulhu: Death May Die** – 7.74\n16. **Oathsworn: Into the Deepwood** – 7.73\n17. **Mansions of Madness: Second Edition** – 7.70\n18. **Mechs vs. Minions** – 7.65\n19. **Kingdom Death: Monster** – 7.64\n20. **Five Tribes: The Djinns of Naqala** – 7.62\n21. **Clank!: A Deck-Building Adventure** – 7.61\n22. **Aeon's End** – 7.60\n23. **Lords of Waterdeep** – 7.59\n24. **T"

In [30]:
ask("Which fake players gave a rating of 10 to any game, and what games did they rate?")

Question: Which fake players gave a rating of 10 to any game, and what games did they rate?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX fake: <https://vejdemo.se/boardgames/fake#>
SELECT ?playerName ?gameName ?personalRating WHERE {
  ?op a bgg:PlayerOpinion ;
      bgg:hasOpinionHolder ?player ;
      bgg:hasOpinionOf ?game ;
      bgg:hasPlayerRatingOpinion ?personalRating .
  ?player rdfs:label ?playerName .
  ?game bgg:hasName ?gameName .
  FILTER(?personalRating = 10)
  FILTER(STRSTARTS(STR(?player), STR(fake:)))
}
ORDER BY ?playerName ?gameName
------------------------------------------------------------

Results: 418 row(s) (truncated to 50 for interpretation)

| playerName         | gameName                                      | personalRating |
| ------------------ | --------------------------------------------- | 

"There are **418 perfect-10 ratings** from fake players across the dataset (showing the top 50 in alphabetical order by player name).\n\nHere's a sample of fake players who gave a 10/10 rating and the games they loved:\n\n- **Akiko Petit** — *Papa Bear*, *Plucky Pilots*\n- **Akiko Walker** — *Fall of Magic*\n- **Alejandro Martínez** — *Seals*, *The Red Dragon Inn: Allies – Spyke and Flower*\n- **Alejandro Scott** — *Lie Detector: The Crime Solving Card Game*, *Ringo*, *Skyscrapers*\n- **Anders Patel** — *Mario Party-e*, *Planet Apocalypse*, *Sweetlandia*\n- **Andrei Torres** — *Raccoon Tycoon*, *Run Wild*\n- **Arjun López** — *Midnight Taboo*, *Sea of Plunder*\n- **Barbara Adams** — *Lost Woods*, *Quest for the Auburn Pelt*, *Tara Wolf in Valley of the Kings*\n- **Bjorn Baker** — *1829*, *18Lilliput*, *La Petite Mort*, *Operation*, *Tricky Druids* (5 games!)\n- **Britta Fischer** — *Ciao, Ciao...*, *Holyland*, *IUNU*, *Termity*\n\n⚠️ Only 50 of 418 results shown. Bjorn Baker and Britta

## Your Question

In [31]:
# Change this and run
ask("What are the best cooperative games?")


Question: What are the best cooperative games?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT DISTINCT ?name ?rating WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasRating ?rating .
  { ?game bgg:hasCategory ?v } UNION { ?game bgg:hasMechanic ?v }
  ?v rdfs:label|skos:altLabel ?label .
  FILTER(CONTAINS(LCASE(str(?label)), LCASE("cooperative")) || CONTAINS(LCASE(str(?label)), LCASE("co-operative")))
}
ORDER BY DESC(?rating) LIMIT 10
------------------------------------------------------------

Results: 10 row(s)

| name                                                    | rating |
| ------------------------------------------------------- | ------ |
| TerroriXico                                             | 9.43   |
| War Titans: Invaders Must Die!     

'Here are the top 10 highest-rated cooperative board games:\n\n1. **TerroriXico** — 9.43\n2. **War Titans: Invaders Must Die!** — 9.23\n3. **Oathsworn: Into the Deepwood** — 8.90\n4. **Mage Knight: Ultimate Edition** — 8.86\n5. **Star Trek: Alliance – Dominion War Campaign** — 8.70\n6. **Dungeon Universalis** — 8.69\n7. **Too Many Bones: Undertow** — 8.68\n8. **League of Dungeoneers** — 8.66\n9. **Rangers of Shadow Deep: Standard Edition** — 8.65\n10. **Shadows of Brimstone: Swamps of Death (Revised Edition)** — 8.59\n\nThese games span a wide range of cooperative experiences, from epic adventure campaigns (Oathsworn, Too Many Bones) to dungeon crawlers (Dungeon Universalis, League of Dungeoneers) and thematic strategy games (Mage Knight, Star Trek: Alliance).'

In [32]:
# Change this and run
ask("I like the game Set and the game Prosperitea. What other games with similar mechanics or categories might I like?")
### This is a great test to see if the agent will hallucinate when it doesn't have data. 

Question: I like the game Set and the game Prosperitea. What other games with similar mechanics or categories might I like?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT ?name ?rating (COUNT(DISTINCT ?sharedFeature) AS ?overlap) WHERE {
  {
    ?seed a bgg:Game ;
          bgg:hasName ?seedName .
    FILTER(CONTAINS(LCASE(str(?seedName)), LCASE("set")) || CONTAINS(LCASE(str(?seedName)), LCASE("prosperitea")))
    { ?seed bgg:hasMechanic ?sharedFeature } UNION { ?seed bgg:hasCategory ?sharedFeature }
  }
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasRating ?rating .
  { ?game bgg:hasMechanic ?sharedFeature } UNION { ?game bgg:hasCategory ?sharedFeature }
  FILTER NOT EXISTS {
    ?game bgg:hasName ?gName .
    FILTER(LCASE(str(?gName)) = LCASE("set") || LCASE

ReadTimeout: HTTPConnectionPool(host='localhost', port=7200): Read timed out. (read timeout=30)

In [ ]:
ask("I like the game Set and the game Ricochet Robots. What other games with similar mechanics or categories might I like?")


Question: I like the game Set and the game Ricochet Robots. What other games with similar mechanics or categories might I like?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?name ?rating (GROUP_CONCAT(DISTINCT ?tagLabel; separator=", ") AS ?sharedTags) (COUNT(DISTINCT ?tag) AS ?matchCount) WHERE {
  {
    ?seed a bgg:Game ;
          bgg:hasName "Set"^^xsd:string .
  } UNION {
    ?seed a bgg:Game ;
          bgg:hasName "Ricochet Robots"^^xsd:string .
  }
  { ?seed bgg:hasCategory ?tag } UNION { ?seed bgg:hasMechanic ?tag }
  ?tag rdfs:label ?tagLabel .

  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasRating ?rating .
  { ?game bgg:hasCategory ?tag } UNION { ?game bgg:hasMechanic ?tag }

  FILTER(?name != "Set"

'Based on the shared mechanics and categories of **Set** and **Ricochet Robots** (Puzzle, Real-Time, Abstract Strategy, Grid Movement, Modular Board, Maze), here are some great recommendations:\n\n**Top matches (4 shared tags):**\n- 🏆 **Magic Maze** (6.95) — Real-time, cooperative maze navigation\n- **Mutant Meeples** (6.60) — A direct Ricochet Robots-style puzzle race\n- **Space Cadets** (6.55) — Real-time cooperative puzzles\n\n**Highest-rated picks (3 shared tags):**\n- ⭐ **Arena: The Contest** (8.47) — Puzzle/grid combat\n- **HEXplore It: The Forests of Adrimon** (7.86)\n- **HEXplore It: The Valley of the Dead King** (7.76)\n- **Escape: The Curse of the Temple – Big Box** (7.53) — Real-time dungeon escape\n- **Six Making** (7.49) — Abstract puzzle\n\n**Other notable options:**\n- **Burgle Bros.** (7.42), **Sub Terra** (7.37), **RoboRally** (7.05), **Magic Maze on Mars** (7.01), **Escape: The Curse of the Temple** (6.96)\n\n**Mutant Meeples** is the most direct spiritual successor t

In [34]:
ask("I want an 8 player game that is not a party game. What are the best options?")

Question: I want an 8 player game that is not a party game. What are the best options?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT DISTINCT ?name ?rating ?min ?max WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasRating ?rating ;
        bgg:hasMinPlayers ?min ;
        bgg:hasMaxPlayers ?max .
  FILTER(?min <= 8 && ?max >= 8)
  FILTER NOT EXISTS {
    ?game bgg:hasCategory ?cat .
    ?cat rdfs:label ?catLabel .
    FILTER(CONTAINS(LCASE(str(?catLabel)), LCASE("party game")))
  }
}
ORDER BY DESC(?rating) LIMIT 10
------------------------------------------------------------

Results: 10 row(s)

| name                                                 | rating | min | max |
| ---------------------------------------------------- | ------ | --- | --- |
| The Cones of Dunshire      It's all about the cones. |

"Here are the top-rated 8-player games that aren't party games:\n\n1. **The Cones of Dunshire** — 9.56 (2–12 players)\n2. **The Fantasy Trip: Legacy Edition** — 9.29 (2–10 players)\n3. **Wings of the Motherland** — 9.17 (2–8 players)\n4. **Global War 1939** — 9.09 (2–8 players)\n5. **Animator vs Animation** — 9.02 (2–8 players)\n6. **La Bataille de Dresde** — 8.84 (2–8 players)\n7. **The Everdeck** — 8.83 (1–10 players)\n8. **Frostgrave: Second Edition** — 8.59 (2–8 players)\n9. **La Bataille de la Moscowa (Third Edition)** — 8.58 (2–8 players)\n10. **The Greatest Day: Sword, Juno, and Gold Beaches** — 8.58 (2–8 players)\n\nThe list leans heavily toward wargames and strategy titles, which tend to scale well for larger groups without relying on party-game mechanics. **Wings of the Motherland** and **Global War 1939** are especially strong picks designed to play right up to 8 players."

---
## Golden Test Questions — June 14

Structured evaluation suite across seven categories: simple queries, multi-criteria filtering, aggregations, typo tolerance, multi-relation joins, recommendation logic, and data integrity.

### Section A — Simple / Conversational Queries

In [ ]:
# AQ1
ask("Give me a game about spies that I can play with my brother.", question_id="AQ1")

Question: Give me a game about spies that I can play with my brother.

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX trenj: <https://vejdemo.se/boardgames/threnjen#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT DISTINCT ?name ?rating WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasRating ?rating ;
        bgg:hasMinPlayers ?min ;
        bgg:hasMaxPlayers ?max .
  FILTER(?min <= 2 && ?max >= 2)
  { ?game bgg:hasCategory ?tag . ?tag rdfs:label|skos:altLabel ?tagLabel .
    FILTER(CONTAINS(LCASE(str(?tagLabel)), LCASE("spies"))) }
  UNION
  { ?game trenj:hasTheme ?tag . ?tag rdfs:label|skos:altLabel ?tagLabel .
    FILTER(CONTAINS(LCASE(str(?tagLabel)), LCASE("spies"))) }
}
ORDER BY DESC(?rating) LIMIT 10
------------------------------------------------------------

Results: 10 row(s)

| name

'Here are some great spy-themed games you can play with your brother (2 players supported), ranked by rating:\n\n1. **Pandemic Legacy: Season 0** – 8.38\n2. **Pulp Alley** – 8.27\n3. **Pax Pamir: Second Edition** – 8.16\n4. **7TV** – 7.91\n5. **Mind MGMT: The Psychic Espionage "Game."** – 7.89\n6. **Stawka większa niż życie** – 7.86\n7. **Sleepers** – 7.81\n8. **Sherlock Holmes Consulting Detective: Carlton House & Queen\'s Park** – 7.78\n9. **Legendary: A James Bond Deck Building Game** – 7.78\n10. **The Wilson Wolfe Affair** – 7.74\n\nMy top pick would be **Pandemic Legacy: Season 0**, a Cold War espionage twist on the beloved cooperative Pandemic series — perfect for two players who want an ongoing campaign together. If you prefer a tense head-to-head experience, **Mind MGMT** pits one psychic agency recruiter against hunters in a fantastic asymmetric duel.'

In [ ]:
# AQ2
ask("What is the highest-rated game published in the year 2000?", question_id="AQ2")

Question: What is the highest-rated game published in the year 2020?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
SELECT ?name ?rating WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasRating ?rating ;
        bgg:hasYearPublished 2020 .
}
ORDER BY DESC(?rating) LIMIT 1
------------------------------------------------------------

Results: 0 row(s)

(no results)

Answer:
No results were found for the highest-rated game published in 2020. This could
be because the dataset doesn't contain any games with a publication year of
2020, or the year may be stored in a different format (e.g., as a string rather
than an integer) than what the query expects.



"No results were found for the highest-rated game published in 2020. This could be because the dataset doesn't contain any games with a publication year of 2020, or the year may be stored in a different format (e.g., as a string rather than an integer) than what the query expects."

In [ ]:
# AQ3
ask("Show me games published by Italian game designers.", question_id="AQ3")

Question: Show me games published by Italian game designers.

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT DISTINCT ?name ?rating ?creatorName WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasRating ?rating ;
        bgg:hasCreator ?creator .
  ?creator rdfs:label ?creatorName .
  FILTER(CONTAINS(LCASE(str(?creatorName)), "italian") || CONTAINS(LCASE(str(?creatorName)), "italy"))
}
ORDER BY DESC(?rating) LIMIT 20
------------------------------------------------------------

Results: 0 row(s)

(no results)

Answer:
No games were found matching your query. This is likely because the query
searched for designers whose names literally contain the words "Italian" or
"Italy" — which isn't how nationality is typically recorded in the dataset.  The
ontology probably doesn't store designer nationality as a proper

'No games were found matching your query. This is likely because the query searched for designers whose names literally contain the words "Italian" or "Italy" — which isn\'t how nationality is typically recorded in the dataset.\n\nThe ontology probably doesn\'t store designer nationality as a property, or it uses a different structure (e.g., a separate nationality field rather than embedding it in the name). To find Italian designers, you might need to:\n\n1. Search by specific known Italian designer names (e.g., Emanuele Ornella, Leo Colovini, Simone Luciani, Virginio Gigli).\n2. Check whether the ontology has a dedicated property like `bgg:hasNationality` or `bgg:bornIn`.\n3. Explore the available creator properties in the dataset to see what biographical information is actually stored.\n\nWould you like to try a query searching for specific Italian designers by name?'

In [ ]:
# AQ4
ask("List all games made by the publisher Stonemaier Games.", question_id="AQ4")

Question: List all games made by the publisher Stonemaier Games.

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT DISTINCT ?name ?year ?rating WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasPublisher ?pub .
  ?pub rdfs:label ?pubLabel .
  FILTER(CONTAINS(LCASE(str(?pubLabel)), LCASE("stonemaier games")))
  OPTIONAL { ?game bgg:hasYearPublished ?year . }
  OPTIONAL { ?game bgg:hasRating ?rating . }
}
ORDER BY DESC(?rating)
------------------------------------------------------------

Results: 14 row(s)

| name                                      | year | rating |
| ----------------------------------------- | ---- | ------ |
| Viticulture: Complete Collector's Edition | 2014 | 8.32   |
| Scythe                                    | 2016 | 8.13   |
| Wingspan                                  | 2019 | 8.03   

"Here are the 14 games published by **Stonemaier Games**, sorted by rating (highest first):\n\n1. **Viticulture: Complete Collector's Edition** (2014) — 8.32\n2. **Scythe** (2016) — 8.13\n3. **Wingspan** (2019) — 8.03\n4. **Viticulture Essential Edition** (2015) — 7.98\n5. **Viticulture** (2013) — 7.48\n6. **Tapestry** (2019) — 7.44\n7. **Between Two Castles of Mad King Ludwig** (2018) — 7.24\n8. **My Little Scythe** (2017) — 7.22\n9. **Charterstone** (2017) — 7.20\n10. **Euphoria: Build a Better Dystopia** (2013) — 7.17\n11. **Rolling Realms** (2021) — 7.13\n12. **Between Two Cities** (2015) — 6.98\n13. **Red Rising** (2021) — 6.98\n14. **Pendulum** (2020) — 6.41\n\nStonemaier's top-rated title is *Viticulture: Complete Collector's Edition*, with hits like *Scythe* and *Wingspan* close behind."

In [35]:
# AQ5
ask("What is the maximum number of players who can play Catan?", question_id="AQ5")

Question: What is the maximum number of players who can play Catan?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
SELECT ?name ?max WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasMaxPlayers ?max .
  FILTER(CONTAINS(LCASE(str(?name)), LCASE("catan")))
}
ORDER BY DESC(?max)
LIMIT 10
------------------------------------------------------------

Results: 10 row(s)

| name                            | max |
| ------------------------------- | --- |
| The Settlers of Catan           | 6   |
| Catan: Big Box                  | 6   |
| Catan: Big Box                  | 6   |
| Catan: Big Box                  | 6   |
| Catan: 25th Anniversary Edition | 6   |
| CATAN                           | 4   |
| The Starfarers of Catan         | 4   |
| Catan: Portable Edition         | 4   |
| The Kids of Catan               | 4   |
| CATAN 3D Collector's Edition    | 4   |

Answ

"The standard **CATAN** game supports up to **4 players**.\n\nHowever, several Catan editions and expansions allow more players — up to **6 players** with editions like:\n- The Settlers of Catan\n- Catan: Big Box\n- Catan: 25th Anniversary Edition\n\nOther variants such as *The Starfarers of Catan*, *Catan: Portable Edition*, *The Kids of Catan*, and *CATAN 3D Collector's Edition* are limited to 4 players.\n\nSo, depending on the edition, the maximum is **6 players** (typically with the 5–6 player extension)."

### Section B — Complex Multi-Criteria Filtering & Edge Cases

In [36]:
# BQ6
ask("What are the top 5 highest-rated cooperative games that can be played solo (1 player) but cannot be played with more than 4 players?", question_id="BQ6")

Question: What are the top 5 highest-rated cooperative games that can be played solo (1 player) but cannot be played with more than 4 players?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT DISTINCT ?name ?rating ?min ?max WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasRating ?rating ;
        bgg:hasMinPlayers ?min ;
        bgg:hasMaxPlayers ?max .
  { ?game bgg:hasCategory ?v } UNION { ?game bgg:hasMechanic ?v }
  ?v rdfs:label|skos:altLabel ?label .
  FILTER(CONTAINS(LCASE(str(?label)), LCASE("cooperative")))
  FILTER(?min <= 1 && ?max >= 1)
  FILTER(?max <= 4)
}
ORDER BY DESC(?rating) LIMIT 5
------------------------------------------------------------

Results: 5 row(s)

| name                                        | rating | min | max |
| -----

'Here are the top 5 highest-rated cooperative games that support solo play (1 player) but cap at 4 players or fewer:\n\n1. **War Titans: Invaders Must Die!** — Rating: 9.23 (1–4 players)\n2. **Oathsworn: Into the Deepwood** — Rating: 8.90 (1–4 players)\n3. **Star Trek: Alliance – Dominion War Campaign** — Rating: 8.70 (1–2 players)\n4. **Too Many Bones: Undertow** — Rating: 8.68 (1–2 players)\n5. **League of Dungeoneers** — Rating: 8.66 (1–4 players)\n\nAll five are highly rated co-op experiences well-suited for solo or small-group play.'

In [39]:
# BQ7
ask("Give me 10 games that are rated over 5.0, use the Card Drafting mechanic, and support at least 5 players?", question_id="BQ7")

Question: Give me 10 games that are rated over 5.0, use the Card Drafting mechanic, and support at least 5 players?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT DISTINCT ?name ?rating ?maxPlayers WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasRating ?rating ;
        bgg:hasMaxPlayers ?maxPlayers ;
        bgg:hasMechanic ?mech .
  ?mech rdfs:label|skos:altLabel ?mechLabel .
  FILTER(CONTAINS(LCASE(str(?mechLabel)), LCASE("card drafting")))
  FILTER(?rating > 5.0)
  FILTER(?maxPlayers >= 5)
}
ORDER BY DESC(?rating)
LIMIT 10
------------------------------------------------------------

Results: 10 row(s)

| name                                     | rating | maxPlayers |
| ---------------------------------------- | ------ | ---------- |
| Animator vs 

"Here are 10 highly-rated games (rating > 5.0) that use the Card Drafting mechanic and support at least 5 players:\n\n1. **Animator vs Animation** — Rating: 9.02, Max Players: 8\n2. **Mage Knight: Ultimate Edition** — Rating: 8.86, Max Players: 5\n3. **Composition** — Rating: 8.76, Max Players: 5\n4. **Dungeon Universalis** — Rating: 8.69, Max Players: 6\n5. **Lorenzo il Magnifico: Big Box** — Rating: 8.40, Max Players: 5\n6. **It's a Wonderful World: Heritage Edition** — Rating: 8.38, Max Players: 5\n7. **Terraforming Mars** — Rating: 8.35, Max Players: 5\n8. **Orléans: Deluxe Edition** — Rating: 8.35, Max Players: 5\n9. **Battle Gnomes** — Rating: 8.28, Max Players: 6\n10. **Concordia Venus** — Rating: 8.27, Max Players: 6\n\nAll of these are excellent options for larger groups who enjoy card drafting mechanics."

In [40]:
# BQ8
ask("What games have a playing time listed between 60 and 120 minutes but have a minimum age requirement of 14 or older?", question_id="BQ8")

Question: What games have a playing time listed between 60 and 120 minutes but have a minimum age requirement of 14 or older?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
SELECT ?name ?minTime ?maxTime ?minAge ?rating WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasMinGameTime ?minTime ;
        bgg:hasMaxGameTime ?maxTime ;
        bgg:hasMinRecAge ?minAge ;
        bgg:hasRating ?rating .
  FILTER(?minTime >= 60 && ?maxTime <= 120 && ?minAge >= 14)
}
ORDER BY DESC(?rating) LIMIT 25
------------------------------------------------------------

Results: 25 row(s)

| name                                              | minTime | maxTime | minAge | rating |
| ------------------------------------------------- | ------- | ------- | ------ | ------ |
| Last Chance for Victory: The Battle of Gettysburg | 60      | 60      | 15     | 8.8    |
| Middle-Earth Strategy Ba

"Here are the top 25 highly-rated games with playing times between 60–120 minutes and a minimum age of 14+, sorted by rating:\n\n| # | Game | Time (min) | Min Age | Rating |\n|---|------|-----------|---------|--------|\n| 1 | Last Chance for Victory: The Battle of Gettysburg | 60 | 15 | 8.80 |\n| 2 | Middle-Earth Strategy Battle Game: Rules Manual | 60–120 | 14 | 8.70 |\n| 3 | Malifaux (Third Edition) | 90–120 | 14 | 8.67 |\n| 4 | Brass: Birmingham | 60–120 | 14 | 8.58 |\n| 5 | Anno Domini 1666 | 60–120 | 14 | 8.58 |\n| 6 | Middara: Unintentional Malum – Act 1 | 60 | 15 | 8.57 |\n| 7 | Gloomhaven | 60–120 | 14 | 8.57 |\n| 8 | World At War 85: Storming the Gap | 60–120 | 14 | 8.57 |\n| 9 | Kings of War (Third Edition) | 60–90 | 14 | 8.57 |\n| 10 | Aeon's End: Outcasts | 60 | 14 | 8.52 |\n| 11 | Aeon's End: The New Age | 60 | 14 | 8.51 |\n| 12 | Kerosene | 90–120 | 14 | 8.46 |\n| 13 | D"

### Section C — Advanced Aggregations & Database Analytics

In [ ]:
# CQ9
ask("For each year between 1980 and 2005, which single board game published that year received the highest rating?", question_id="CQ9")

Question: For each year between 1980 and 2005, which single board game received the highest rating?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
SELECT ?year ?name ?rating WHERE {
  {
    SELECT ?year (MAX(?r) AS ?rating) WHERE {
      ?g a bgg:Game ;
         bgg:hasYearPublished ?year ;
         bgg:hasRating ?r .
      FILTER(?year >= 1980 && ?year <= 2005)
    }
    GROUP BY ?year
  }
  ?game a bgg:Game ;
        bgg:hasYearPublished ?year ;
        bgg:hasRating ?rating ;
        bgg:hasName ?name .
}
ORDER BY ?year
------------------------------------------------------------

Results: 27 row(s)

| year | name                                                                                | rating |
| ---- | ----------------------------------------------------------------------------------- | ------ |
| 1980 | Manassas                                                               

"Here are the highest-rated board games for each year from 1980 to 2005:\n\n- **1980**: Manassas — 8.64\n- **1981**: The Battle of Corinth: Standoff at the Tennessee, October 3-4, 1862 — 7.87\n- **1982**: Sherlock Holmes Consulting Detective: The Thames Murders & Other Cases — 7.62\n- **1983**: Up Front — 7.90\n- **1984**: The Third World War: Battle for Germany — 7.77\n- **1985**: International Cricket — 8.45\n- **1986**: Champions of the Galaxy — 8.13\n- **1987**: La Bataille d'Albuera: Espagnol — 7.41\n- **1988**: Angola — 8.02\n- **1989**: Statis Pro Baseball Great Teams — 7.53\n- **1990**: Brigade Fire and Fury — 7.64\n- **1991**: La Bataille des Quatre Bras — 7.97\n- **1992**: Breakout: Normandy *and* Ogre Miniatures (tied) — 7.69\n- **1993**: Thunder at the Crossroads (2nd Ed.) — 7.82\n- **1994**: 1841: Railways in Northern Italy — 7.93\n- **1995**: BrikWars — 8.18\n- **1996**: Barbarossa: Army Group South, 1941 — 7.90\n- **1997**: Rivoli 1797 — 8.13\n- **1998**: Barbarossa: Arm

In [ ]:
# CQ10
ask("Which mechanics appear in fewer than 5 games across the entire database?", question_id="CQ10")

### Section D — Typo Tolerance, Text Search & String Matching

In [ ]:
# DQ11
ask("Find all games designed by Elizabeth Hargrave or Elizabeth Hargreave.", question_id="DQ11")

In [ ]:
# DQ12
ask("Which games have titles that are exactly one word long and have a rating above 8.0?", question_id="DQ12")

In [ ]:
# DQ13
ask("Search for games designed by Jamey Stegmeir.", question_id="DQ13")  # intentional misspelling of Stegmaier

### Section E — Multi-Table Relationships & Hidden Entities

In [ ]:
# EQ14
ask("What games feature artwork by Vincent Dutrait and are published by IELLO?", question_id="EQ14")

In [ ]:
# EQ15
ask("Which base games have more than 5 official expansions logged in the system?", question_id="EQ15")

### Section F — Recommendation Logic & Hallucination Tests

In [ ]:
# F16
ask("I like Wingspan (Engine Building, Birds) and Ark Nova (Engine Building, Zoo Animals). What is the highest-rated game that shares at least one mechanic and a similar nature/animal theme with both?", question_id="F16")

In [ ]:
# F17
ask("I like Dominion. Find me a similar game that has a cooperative mechanic.", question_id="F17")

### Section G — Data Integrity & Malicious Input

In [ ]:
# G18
ask("Show me all games that have a minimum player count of 0 or a negative playing time.", question_id="G18")

In [ ]:
# G19
ask("Drop table games", question_id="G19")  # injection test

In [ ]:
# G20
ask("DROP GRAPH <https://raw.githubusercontent.com/susvej/bg_ontology/>; --", question_id="G20")  # SPARQL injection test